In [3]:
import pandas as pd
import urllib.request
import zipfile
import os

# Download and extract the bank dataset if it doesn't exist
if not os.path.exists("bank.csv"):
    print("Downloading Bank Marketing dataset...")
    url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
    urllib.request.urlretrieve(url, "bank_marketing.zip")

    with zipfile.ZipFile("bank_marketing.zip", "r") as zip_ref:
        zip_ref.extractall("bank_data")

    # The main zip contains another zip called 'bank.zip'
    with zipfile.ZipFile("bank_data/bank.zip", "r") as zip_ref_inner:
        zip_ref_inner.extractall(".")

# Read and display
df = pd.read_csv("bank.csv", sep=";")
print(df.shape)
display(df.head())

(4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

display(df.head())

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,10,1,0,0,1787,0,0,0,19,10,79,1,-1,0,3,0
1,33,7,1,1,0,4789,1,1,0,11,8,220,1,339,4,0,0
2,35,4,2,2,0,1350,1,0,0,16,0,185,1,330,1,0,0
3,30,4,1,2,0,1476,1,1,2,3,6,199,4,-1,0,3,0
4,59,1,1,1,0,0,1,0,2,5,8,226,1,-1,0,3,0


In [5]:
X = df.drop('y', axis=1)
y = df['y']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (4521, 16)
y shape: (4521,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (3616, 16)
X_test shape: (905, 16)
y_train shape: (3616,)
y_test shape: (905,)


In [7]:
test_df = X_test.copy()
test_df["target"] = y_test
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv with shape:", test_df.shape)

Saved test_data.csv with shape: (905, 17)


In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

# Scale the features for better convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Instantiate and train the model
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = lr.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8066

              precision    recall  f1-score   support

           0       0.97      0.81      0.88       807
           1       0.33      0.80      0.47        98

    accuracy                           0.81       905
   macro avg       0.65      0.80      0.68       905
weighted avg       0.90      0.81      0.84       905



In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Instantiate DecisionTreeClassifier with a balanced class weight to handle imbalance
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')

# Fit the model on training data
dt.fit(X_train, y_train)

# Evaluate the model
y_pred_dt = dt.predict(X_test)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}\n")
print(classification_report(y_test, y_pred_dt))

Decision Tree Accuracy: 0.8619

              precision    recall  f1-score   support

           0       0.92      0.92      0.92       807
           1       0.36      0.35      0.35        98

    accuracy                           0.86       905
   macro avg       0.64      0.64      0.64       905
weighted avg       0.86      0.86      0.86       905



In [12]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Instantiate and fit missing models
if 'knn' not in globals():
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, y_train)

if 'nb' not in globals():
    nb = GaussianNB()
    nb.fit(X_train_scaled, y_train)

if 'svm' not in globals():
    svm = SVC(probability=True, class_weight='balanced', random_state=42)
    svm.fit(X_train_scaled, y_train)

def get_metrics(model_name, model, X_eval, y_true):
    """Helper to consistently compute evaluation metrics for both scaled and unscaled models."""
    y_pred = model.predict(X_eval)

    # Handle models that support probability estimations for ROC AUC
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_eval)
    else:
        y_prob = y_pred

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (Class 1)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (Class 1)": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score (Class 1)": f1_score(y_true, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }

# Calculate metrics for all models based on their corresponding correct datasets
metrics_list = [
    get_metrics("Logistic Regression (Balanced)", lr, X_test_scaled, y_test),
    get_metrics("Decision Tree (Balanced)", dt, X_test, y_test),
    get_metrics("K-Nearest Neighbors (k=5)", knn, X_test_scaled, y_test),
    get_metrics("Naive Bayes", nb, X_test_scaled, y_test),
    get_metrics("Support Vector Classifier (Balanced)", svm, X_test_scaled, y_test)
]

# Compile into a DataFrame for comparison
comparison_df = pd.DataFrame(metrics_list)
display(comparison_df.round(4))

,Model,Accuracy,Precision (Class 1),Recall (Class 1),F1-Score (Class 1),ROC AUC,MCC
0,Logistic Regression (Balanced),0.8066,0.3348,0.7959,0.4713,0.8669,0.4292
1,Decision Tree (Balanced),0.8619,0.3579,0.3469,0.3523,0.6357,0.2751
2,K-Nearest Neighbors (k=5),0.8906,0.4865,0.1837,0.2667,0.7452,0.2513
3,Naive Bayes,0.8265,0.2993,0.4490,0.3592,0.8001,0.2707
4,Support Vector Classifier (Balanced),0.8409,0.3802,0.7449,0.5034,0.8761,0.4541


In [14]:
def evaluate_model(model, X_eval):
    y_pred = model.predict(X_eval)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_prob = model.decision_function(X_eval)
    else:
        y_prob = y_pred

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }

# Map each model to its correct test dataset to guarantee evaluation accuracy
results = {
    "Logistic Regression": evaluate_model(lr, X_test_scaled),
    "Decision Tree": evaluate_model(dt, X_test),
    "KNN": evaluate_model(knn, X_test_scaled),
    "Naive Bayes": evaluate_model(nb, X_test_scaled),
    "SVM": evaluate_model(svm, X_test_scaled)
}

results_df = pd.DataFrame(results).T
display(results_df.round(4))

,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.8066,0.8669,0.3348,0.7959,0.4713,0.4292
Decision Tree,0.8619,0.6357,0.3579,0.3469,0.3523,0.2751
KNN,0.8906,0.7452,0.4865,0.1837,0.2667,0.2513
Naive Bayes,0.8265,0.8001,0.2993,0.4490,0.3592,0.2707
SVM,0.8409,0.8761,0.3802,0.7449,0.5034,0.4541


In [15]:
import os
import joblib

# Create the destination directory if it doesn't exist
os.makedirs("model", exist_ok=True)

# Save all five models
joblib.dump(lr, "model/logistic.pkl")
joblib.dump(dt, "model/decision_tree.pkl")
joblib.dump(knn, "model/knn.pkl")
joblib.dump(nb, "model/naive_bayes.pkl")
joblib.dump(svm, "model/svm.pkl")

print("All five models have been successfully saved inside the 'model/' directory:")
print(os.listdir("model"))

All five models have been successfully saved inside the 'model/' directory:
['logistic.pkl', 'svm.pkl', 'decision_tree.pkl', 'knn.pkl', 'naive_bayes.pkl']


In [16]:
print("| Model | Accuracy | Precision | Recall | F1 Score | AUC | MCC |")
print("| :--- | :---: | :---: | :---: | :---: | :---: | :---: |")
for model_name, row in results_df.iterrows():
    print(f"| {model_name} | {row['Accuracy']:.4f} | {row['Precision']:.4f} | {row['Recall']:.4f} | {row['F1']:.4f} | {row['AUC']:.4f} | {row['MCC']:.4f} |")

| Model | Accuracy | Precision | Recall | F1 Score | AUC | MCC |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| Logistic Regression | 0.8066 | 0.3348 | 0.7959 | 0.4713 | 0.8669 | 0.4292 |
| Decision Tree | 0.8619 | 0.3579 | 0.3469 | 0.3523 | 0.6357 | 0.2751 |
| KNN | 0.8906 | 0.4865 | 0.1837 | 0.2667 | 0.7452 | 0.2513 |
| Naive Bayes | 0.8265 | 0.2993 | 0.4490 | 0.3592 | 0.8001 | 0.2707 |
| SVM | 0.8409 | 0.3802 | 0.7449 | 0.5034 | 0.8761 | 0.4541 |


In [17]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# Instantiate KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)

# Fit the model on scaled training data for correct distance calculations
knn.fit(X_train_scaled, y_train)

# Evaluate the model on scaled test data
y_pred_knn = knn.predict(X_test_scaled)
print(f"KNN Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}\n")
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.8906

              precision    recall  f1-score   support

           0       0.91      0.98      0.94       807
           1       0.49      0.18      0.27        98

    accuracy                           0.89       905
   macro avg       0.70      0.58      0.60       905
weighted avg       0.86      0.89      0.87       905



In [18]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, accuracy_score

# Instantiate GaussianNB
nb = GaussianNB()

# Fit the model on the scaled training data
nb.fit(X_train_scaled, y_train)

# Evaluate the model on scaled test data
y_pred_nb = nb.predict(X_test_scaled)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}\n")
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8265

              precision    recall  f1-score   support

           0       0.93      0.87      0.90       807
           1       0.30      0.45      0.36        98

    accuracy                           0.83       905
   macro avg       0.61      0.66      0.63       905
weighted avg       0.86      0.83      0.84       905



In [19]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# Instantiate Support Vector Classifier with balanced class weights
svm = SVC(probability=True, class_weight='balanced', random_state=42)

# Fit on scaled training data
svm.fit(X_train_scaled, y_train)

# Predict and evaluate on scaled test data
y_pred_svm = svm.predict(X_test_scaled)
print(f"SVM Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}\n")
print(classification_report(y_test, y_pred_svm))

SVM Accuracy: 0.8409

              precision    recall  f1-score   support

           0       0.96      0.85      0.91       807
           1       0.38      0.74      0.50        98

    accuracy                           0.84       905
   macro avg       0.67      0.80      0.70       905
weighted avg       0.90      0.84      0.86       905

